# Large-scale OR: GPT-4.1 only

One GPT-4.1 call per case receives the current question and its complete CSV files. No classification call, examples, or retrieval tools. The model returns an LP/MILP formulation and executable gurobipy code.


In [ ]:
from __future__ import annotations

import csv, json, math, os, re, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path(os.environ.get('LEAN_LLM_OPT_ROOT', Path.cwd())).resolve()
BENCHMARK = ROOT / 'Test_Dataset/Large-scale-or/Large-scale-or-101.csv'
MODEL = 'gpt-4.1-2025-04-14'
if not BENCHMARK.is_file():
    raise FileNotFoundError('Run from the repository root or set LEAN_LLM_OPT_ROOT')

OUTPUT_RULES = '''Return exactly two sections:
### Mathematical Model
Specify sets, parameters, variables and their domains, objective, and all constraints.
Use an LP or MILP as required by the current question.
### Python Code
Provide exactly one fenced Python code block. It must be complete gurobipy
code, create a model named m or model, call optimize(), and leave that solved
model available at top level. Read only the current case CSV paths listed in
this request. Derive identifiers, dimensions, and coefficients from these files.
The code must solve from the repository root; do not invent file paths.'''

def resolve_csv_paths(address):
    out=[]
    for raw in str(address).splitlines():
        raw=raw.strip()
        if not raw:continue
        path=Path(raw)
        path=(path if path.is_absolute() else ROOT/path).resolve()
        try:path.relative_to(ROOT)
        except ValueError as exc:raise ValueError(f'Path leaves repository: {raw}') from exc
        if path.suffix.lower()!='.csv' or not path.is_file():
            raise FileNotFoundError(path)
        out.append(path)
    if not out:raise ValueError('No CSV files listed for this case')
    return out

def parse_objective(raw):
    text=str(raw).strip().replace(',','').replace('$','')
    m=re.search(r'Optimal\s+objective\s*[:=]?\s*([-+\d.eE]+)',text,re.I)
    value=float(m.group(1) if m else text)
    if not math.isfinite(value):raise ValueError(raw)
    return value

def gold_class(raw):
    text=str(raw).strip().upper()
    if text=='UFLP':return 'FLP'
    if text.startswith('OTHERS'):return 'Others'
    if text=='MIXTURE':return 'Mixture'
    if text in {'NRM','RA','TP','FLP','AP'}:return text
    raise ValueError(raw)

def cases():
    with BENCHMARK.open(encoding='utf-8-sig',newline='') as f:
        rows=list(csv.DictReader(f))
    if len(rows)!=101:raise ValueError(f'Expected 101 cases, found {len(rows)}')
    result=[]
    for i,row in enumerate(rows):
        result.append({'row':i,'query':row['Query'],'type':row['Problem Type'],
                       'files':resolve_csv_paths(row['Dataset_address']),
                       'gold':parse_objective(row['Label-objective'])})
    return result

def rel(path):return str(path.relative_to(ROOT))

def check_files():
    data=cases()
    counts={n:sum(len(c['files'])==n for c in data) for n in (1,2,3)}
    print('Rows:',len(data),'Files per row:',counts)
    return data

def output_text(response):
    if getattr(response,'status',None)=='incomplete':
        raise RuntimeError(f'Incomplete model response: {response.incomplete_details}')
    text=str(getattr(response,'output_text','') or '').strip()
    if not text:raise RuntimeError('Empty model response')
    return text

def usage(response):
    u=getattr(response,'usage',None)
    return {'input':int(getattr(u,'input_tokens',0) or 0),
            'output':int(getattr(u,'output_tokens',0) or 0)}

def extract_code(answer):
    if '### Mathematical Model' not in answer or '### Python Code' not in answer:
        raise ValueError('Model/code sections missing')
    blocks=re.findall(r'```(?:python|py)\s*\n(.*?)\n```',answer,re.I|re.S)
    if len(blocks)!=1:raise ValueError(f'Expected one Python block, found {len(blocks)}')
    return blocks[0].strip()

SOLVER_FOOTER = '''
import gurobipy as _ab_gp
_ab_model=globals().get('m',globals().get('model'))
if not isinstance(_ab_model,_ab_gp.Model):
    raise RuntimeError('Code did not leave a Gurobi model in m or model')
if _ab_model.Status!=_ab_gp.GRB.OPTIMAL:
    raise RuntimeError(f'Gurobi status: {_ab_model.Status}')
print('ABLATION_OBJECTIVE='+repr(float(_ab_model.ObjVal)))
'''

def solve_code(source,timeout=300):
    run=subprocess.run([sys.executable,'-c',source+'\n'+SOLVER_FOOTER],
        cwd=ROOT,text=True,capture_output=True,timeout=timeout,check=False)
    transcript=(run.stdout or '')+'\n'+(run.stderr or '')
    if run.returncode:raise RuntimeError(transcript[-3000:])
    match=re.search(r'ABLATION_OBJECTIVE=([-+\d.eE]+)',transcript)
    if not match:raise RuntimeError('Objective missing: '+transcript[-3000:])
    return float(match.group(1)),transcript[-3000:]

def save_scores(folder,selected):
    latest={}
    for line in (folder/'records.jsonl').read_text(encoding='utf-8').splitlines():
        if line.strip():
            record=json.loads(line);latest[record['row']]=record
    rows=[latest[i] for i in sorted(latest)]
    fields=['run_id','method','dataset','instance_id','gold_type','pred_type',
            'correct_optimal','gold_objective','predicted_objective','status',
            'dataset_address','input_tokens','output_tokens','tool_calls','wall_s']
    with (folder/'scored.csv').open('w',encoding='utf-8',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader()
        for r in rows:
            writer.writerow({'run_id':folder.name,'method':METHOD,
                'dataset':'Large-Scale-OR','instance_id':r['row'],
                'gold_type':r['gold_type'],'pred_type':r.get('pred_type'),
                'correct_optimal':int(bool(r.get('correct_optimal'))),
                'gold_objective':r['gold_objective'],
                'predicted_objective':r.get('predicted_objective'),
                'status':r['status'],'dataset_address':'\n'.join(r['files']),
                'input_tokens':(r.get('usage') or {}).get('input',0),
                'output_tokens':(r.get('usage') or {}).get('output',0),
                'tool_calls':r.get('tool_calls',0),'wall_s':r['wall_s']})
    summary={'run_id':folder.name,'method':METHOD,'n':len(rows),
        'complete':set(latest)==set(selected),
        'correct_optimal':sum(bool(r.get('correct_optimal')) for r in rows),
        'classification_correct':None,
        'failed':sum(r['status']!='ok' for r in rows),
        'input_tokens':sum((r.get('usage') or {}).get('input',0) for r in rows),
        'output_tokens':sum((r.get('usage') or {}).get('output',0) for r in rows)}
    (folder/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')


In [ ]:
METHOD = 'Abl-GPT41Only'

MODEL_RULES = (
    'Solve the current operations research question using only its attached CSV files. '
    'No tools are provided. Use the repository paths supplied in the request.\n\n'
    + OUTPUT_RULES
)

def upload_text_file(client, path, cache):
    import tempfile
    if path in cache:
        return cache[path]
    # Attach complete CSV contents as text, preserving rows past the CSV preview limit.
    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', prefix='or_data_',
                                     encoding='utf-8', delete=False) as tmp:
        temp_path = Path(tmp.name)
        with path.open(encoding='utf-8-sig') as original:
            for line in original:
                tmp.write(line)
    try:
        with temp_path.open('rb') as data:
            uploaded = client.files.create(file=data, purpose='user_data')
        cache[path] = uploaded.id
        return uploaded.id
    finally:
        temp_path.unlink(missing_ok=True)

def answer_case(client, case, cache):
    content = [{'type':'input_text', 'text':
        'Current question:\n' + case['query'] + '\n'
        'The following attached CSV files belong to this question. '
        'Read their full contents and use these exact repository paths in the code.'}]
    for path in case['files']:
        content.append({'type':'input_text', 'text':'Current CSV: ' + rel(path)})
        content.append({'type':'input_file', 'file_id':upload_text_file(client,path,cache)})
    response = client.responses.create(
        model=MODEL, temperature=0, max_output_tokens=32768,
        input=[{'role':'system','content':MODEL_RULES},
               {'role':'user','content':content}])
    return output_text(response), usage(response)

def run(rows=None, output_dir=None):
    from dotenv import load_dotenv
    from openai import OpenAI
    load_dotenv(ROOT/'.env', override=False)
    if not os.environ.get('OPENAI_API_KEY'):
        raise RuntimeError('OPENAI_API_KEY is missing')
    selected = set(range(101) if rows is None else rows)
    if not selected.issubset(set(range(101))):
        raise ValueError('Rows must be 0..100')
    all_cases = cases()
    stamp = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
    folder = Path(output_dir or ROOT/'outputs'/'large_scale_or_ablations'/f'gpt41only__{stamp}').resolve()
    folder.mkdir(parents=True,exist_ok=True)
    (folder/'answers').mkdir(exist_ok=True)
    log = folder/'records.jsonl'
    done = {json.loads(x)['row'] for x in log.read_text().splitlines() if x.strip()} if log.exists() else set()
    client = OpenAI(max_retries=0, timeout=180)
    cache = {}
    try:
        for case in all_cases:
            i = case['row']
            if i not in selected or i in done:
                continue
            rec = {'row':i, 'gold_type':gold_class(case['type']),
                   'gold_objective':case['gold'],
                   'files':[rel(p) for p in case['files']],
                   'status':'error', 'tool_calls':0, 'n_demonstrations':0,
                   'model_calls':1}
            t = time.perf_counter()
            try:
                answer, model_usage = answer_case(client,case,cache)
                rec['usage'] = model_usage
                (folder/'answers'/f'row_{i:03d}.txt').write_text(answer,encoding='utf-8')
                source = extract_code(answer)
                objective, solver = solve_code(source)
                rec.update(status='ok', predicted_objective=objective,
                    correct_optimal=math.isclose(objective,case['gold'],rel_tol=1e-4,abs_tol=1e-4),
                    solver_tail=solver)
            except Exception as exc:
                rec.update(correct_optimal=False, error_type=type(exc).__name__,
                           error=str(exc)[-3000:])
            rec['wall_s'] = round(time.perf_counter()-t,3)
            with log.open('a',encoding='utf-8') as f:
                f.write(json.dumps(rec,ensure_ascii=False)+'\n')
            print('row',i,rec['status'],'correct',rec['correct_optimal'])
    finally:
        for file_id in cache.values():
            try:
                client.files.delete(file_id)
            except Exception as exc:
                print('Could not clean uploaded file',file_id,exc)
    save_scores(folder,selected)
    return folder


## Preflight

Check all 101 question-to-file mappings without calling the API.


In [ ]:
DATA = check_files()
print("Model:", MODEL, "Method:", METHOD)


## Run

Set `RUN_API=True` to run all 101 cases, or set `ROWS` to row indices. Reusing `OUTPUT_DIR` skips every logged row, including failures.


In [ ]:
RUN_API = False
ROWS = None
OUTPUT_DIR = None
if RUN_API:
    RUN_DIR = run(rows=ROWS, output_dir=OUTPUT_DIR)
    print(RUN_DIR)
